### Load data and do a quick check

In [1]:
import json
import pandas as pd
with open("../data/raw/batdongsan.txt", encoding="utf-8") as file:
    df = pd.DataFrame(json.load(file))

In [76]:
df.head(3)

,Loại hình nhà ở,Diện tích đất,Tổng số tầng,Giấy tờ pháp lý,address,price,city,Số phòng ngủ,Số phòng vệ sinh,Hướng ban công,Hướng cửa chính,Dự án,Tầng số
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN
2,Biệt thự,"1.100 m²(20,0x50,0)",3,Sổ hồng,"105, Đường Trần Văn Kiểu, Phường 10, Quận 6, T...",300.0,ho-chi-minh,8 phòng,10 WC,NaN,NaN,NaN,NaN


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Loại hình nhà ở   5414 non-null   object
 1   Diện tích đất     5414 non-null   object
 2   Tổng số tầng      3352 non-null   object
 3   Giấy tờ pháp lý   5414 non-null   object
 4   address           5414 non-null   object
 5   price             5414 non-null   object
 6   city              5414 non-null   object
 7   Số phòng ngủ      5008 non-null   object
 8   Số phòng vệ sinh  4631 non-null   object
 9   Hướng ban công    463 non-null    object
 10  Hướng cửa chính   2804 non-null   object
 11  Dự án             110 non-null    object
 12  Tầng số           1 non-null      object
dtypes: object(13)
memory usage: 550.0+ KB


### Basic cleaning (before merging and more preprocessing)
Rename columns

In [2]:
rename_map = {
    "Loại hình nhà ở":"property_type",
    "Diện tích đất":"area",
    "Tổng số tầng":"n_floors",
    "Giấy tờ pháp lý":"legal_docs",
    "Số phòng ngủ":"n_bedrooms",
    "Số phòng vệ sinh":"n_bathrooms",
    "Hướng ban công":"balcony_direction",
    "Hướng cửa chính":"facing_direction",
    "Dự án":"project",
    "Tầng số":"floor_num",
    "city":"city/district"
}
df.rename(rename_map, axis=1, inplace=True)
df.head(2)

,property_type,area,n_floors,legal_docs,address,price,city/district,n_bedrooms,n_bathrooms,balcony_direction,facing_direction,project,floor_num
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN


Remove unusable columns (>50% missing or redundant)

In [3]:
mostly_empty_cols = ['balcony_direction', 'project', 'floor_num']
redundant_cols = ['address']        # only city is enough
df.drop(mostly_empty_cols + redundant_cols, axis=1, inplace=True)

Extract the numeric values for numeric columns

In [3]:
import re

def extract_numeric(s:str|None, thousands_sep:bool=True) -> float:
    if not isinstance(s,str) or not s:       # Handle np.nan (a float), or empty strings
        return None                        # Return np.nan cuz None is treated like a value

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None

    num_str = results.group(1)
    if thousands_sep:
        num_str = num_str.replace(".","")
    num_str = num_str.replace(",",".")

    return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

def extract_dimensions(area_raw:str):
    if not isinstance(area_raw,str) or not area_raw:
        return None

    results = re.search(r"\((.*)x(.*)\)", area_raw)
    if not results:
        return pd.Series((None, None))
    return pd.Series((
        extract_numeric(results.group(1), thousands_sep=False), 
        extract_numeric(results.group(2), thousands_sep=False)
    ))

In [4]:
df[['dimension_1', 'dimension_2']] = df['area'].apply(extract_dimensions)
df['area_num'] = df['area'].apply(extract_numeric)
df['n_bedrooms_2'] = df['n_bedrooms'].apply(extract_numeric)
df['n_bathrooms_2'] = df['n_bathrooms'].apply(extract_numeric)
df['price_2'] = pd.to_numeric(df['price'], errors='coerce', downcast='float') * 1000    # to million
df['n_floors_2'] = pd.to_numeric(df['n_floors'], errors='coerce')

df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2']].sample(10)

,area,area_num,dimension_1,dimension_2,n_bedrooms,n_bedrooms_2,n_bathrooms,n_bathrooms_2,price,price_2,n_floors,n_floors_2
1480,"100 m²(5,0x20,0)",100.0,5.0,20.0,2 phòng,2.0,1 WC,1.0,4.1,4100.0,NaN,NaN
1438,"120 m²(6,0x20,0)",120.0,6.0,20.0,4 phòng,4.0,4 WC,4.0,17.5,17500.0,3,3.0
3710,"99,2 m²(4,0x25,0)",99.2,4.0,25.0,3 phòng,3.0,2 WC,2.0,3.1,3100.0,NaN,NaN
948,"11,8 m²",11.8,NaN,NaN,1 phòng,1.0,1 WC,1.0,1.9,1900.0,2,2.0
1652,"100 m²(5,6x18,4)",100.0,5.6,18.4,2 phòng,2.0,NaN,NaN,3.9,3900.0,1,1.0
3961,"3.000 m²(30,0x100,0)",3000.0,30.0,100.0,2 phòng,2.0,1 WC,1.0,0.37,370.0,NaN,NaN
4316,"100 m²(5,0x20,0)",100.0,5.0,20.0,4 phòng,4.0,4 WC,4.0,7.0,7000.0,4,4.0
3316,"242 m²(12,0x22,0)",242.0,12.0,22.0,11 phòng,11.0,NaN,NaN,4.2,4200.0,NaN,NaN
880,"77 m²(4,2x21,5)",77.0,4.2,21.5,9 phòng,9.0,7 WC,7.0,30.3,30300.0,6,6.0
4959,"85 m²(5,0x15,0)",85.0,5.0,15.0,4 phòng,4.0,3 WC,3.0,3.5,3500.0,3,3.0


In [6]:
df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2'
]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   area           5414 non-null   object 
 1   area_num       5414 non-null   float64
 2   dimension_1    4692 non-null   float64
 3   dimension_2    4692 non-null   float64
 4   n_bedrooms     5008 non-null   object 
 5   n_bedrooms_2   5008 non-null   float64
 6   n_bathrooms    4631 non-null   object 
 7   n_bathrooms_2  4631 non-null   float64
 8   price          5414 non-null   object 
 9   price_2        5406 non-null   float32
 10  n_floors       3352 non-null   object 
 11  n_floors_2     3352 non-null   float64
dtypes: float32(1), float64(6), object(5)
memory usage: 486.5+ KB


In [7]:
df.loc[df.price_2.isna()]

,property_type,area,n_floors,legal_docs,price,city/district,n_bedrooms,n_bathrooms,facing_direction,dimension_1,dimension_2,area_num,n_bedrooms_2,n_bathrooms_2,price_2,n_floors_2
13,Nhà mặt tiền,40 m²,2,Sổ đỏ,thỏa thuận,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,NaN,2.0
97,Nhà mặt tiền,"49,7 m²(4,0x12,0)",NaN,Sổ hồng,thỏa thuận,ho-chi-minh,3 phòng,1 WC,NaN,4.0,12.0,49.7,3.0,1.0,NaN,NaN
280,Nhà hẻm ngõ,"315 m²(8,5x16,4)",2,Sổ hồng,thỏa thuận,ho-chi-minh,NaN,NaN,NaN,8.5,16.4,315.0,NaN,NaN,NaN,2.0
1023,Nhà mặt tiền,80 m²,NaN,Sổ đỏ,thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN
1102,Nhà hẻm ngõ,43 m²,NaN,Sổ đỏ,thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,NaN
1114,Nhà hẻm ngõ,80 m²,5,Sổ đỏ,thỏa thuận,ha-noi,7 phòng,NaN,NaN,NaN,NaN,80.0,7.0,NaN,NaN,5.0
4026,Biệt thự,"56 m²(4,0x14,0)",1,Sổ hồng,thỏa thuận,binh-thuan,2 phòng,2 WC,Tây,4.0,14.0,56.0,2.0,2.0,NaN,1.0
5405,Biệt thự,180 m²,2,Sổ đỏ,thỏa thuận,vinh-phuc,NaN,NaN,NaN,NaN,NaN,180.0,NaN,NaN,NaN,2.0


Observation:
- Non-null counts of `price_2` is lower than the original `price`: "Negotiable" prices 

In [5]:
df.legal_docs.unique()
# df.loc[df.legal_docs == 'Giấy tờ khác'] = 'Khác'

array(['Sổ hồng', 'Sổ đỏ', 'Giấy tờ hợp lệ', 'Giấy tờ khác',
       'Hợp đồng mua bán', 'Đang chờ sổ'], dtype=object)

### Export the file with extracted features

In [ ]:
df.info()

In [9]:
df_final = df[['property_type', 'price_2', 'area_num', 'n_bedrooms_2', 'n_bathrooms_2',
                'n_floors_2', 'dimension_2', 'address', 'legal_docs', 'city/district', 'facing_direction', 'dimension_1']]
df_final = df_final.rename({
    'price_2': 'price',
    'area_num': 'area',
    'n_bedrooms_2': 'n_bedrooms',
    'n_bathrooms_2': 'n_bathrooms', 
    'dimension_1': 'front_width',
    'n_floors_2': 'n_floors'
}, axis=1)

In [10]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   property_type     5414 non-null   object 
 1   price             5406 non-null   float32
 2   area              5414 non-null   float64
 3   n_bedrooms        5008 non-null   float64
 4   n_bathrooms       4631 non-null   float64
 5   n_floors          3352 non-null   float64
 6   dimension_2       4692 non-null   float64
 7   address           5414 non-null   object 
 8   legal_docs        5414 non-null   object 
 9   city/district     5414 non-null   object 
 10  facing_direction  2804 non-null   object 
 11  front_width       4692 non-null   float64
dtypes: float32(1), float64(6), object(5)
memory usage: 486.5+ KB


In [11]:
df_final.to_csv('../data/interim/muaban_net.csv')